# 01 — Document processing pipeline architecture

> **Applied Labs** · **Domain**: MM · **Difficulty**: Advanced · **Reading time**: ~35 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/05_document_processing/01_architecture.ipynb)

**Prerequisites**:
- [00_first_principles.ipynb](./00_first_principles.ipynb)

**What you will learn**:
- End-to-end system architecture for a VLM-based document processing pipeline
- Document classification strategies (invoice, receipt, form, report)
- Layout analysis: detecting headers, tables, and key-value regions
- VLM prompt design for structured extraction with few-shot examples
- Building a rule-based validation pipeline for extracted data
- Confidence scoring and routing logic

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "Pillow>=10.0.0" "pandas>=2.0.0"

import os, json, textwrap, re
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Any, Callable
from enum import Enum
import pandas as pd

print("Setup complete ✓")

## Section 1 — System architecture

The document processing pipeline has six stages, each with clear inputs and outputs:

```
┌──────────┐   ┌──────────────┐   ┌──────────────┐   ┌───────────────┐
│  Ingest   │──▶│  Classify     │──▶│  Layout       │──▶│  VLM Extract  │
│  (image)  │   │  (type)       │   │  Analysis     │   │  (fields)     │
└──────────┘   └──────────────┘   └──────────────┘   └───────────────┘
                                                              │
                                                              ▼
                                  ┌──────────────┐   ┌───────────────┐
                                  │  Route        │◀──│  Validate     │
                                  │  (decision)   │   │  (rules)      │
                                  └──────────────┘   └───────────────┘
                                        │
                        ┌───────────────┼───────────────┐
                        ▼               ▼               ▼
                   Auto-approve    Quick review     Manual entry
```

Each stage is a **pure function** with typed inputs/outputs, making the pipeline
testable and composable.

In [ ]:
# Pipeline stage definitions as typed interfaces

class DocType(Enum):
    """Supported document types."""
    INVOICE = "invoice"
    RECEIPT = "receipt"
    PURCHASE_ORDER = "purchase_order"
    FORM = "form"
    REPORT = "report"

class RouteDecision(Enum):
    """Routing outcomes."""
    AUTO_APPROVE = "auto_approve"
    QUICK_REVIEW = "quick_review"
    MANUAL_ENTRY = "manual_entry"

@dataclass
class PipelineStage:
    """Definition of a pipeline stage."""
    name: str
    input_type: str
    output_type: str
    description: str
    latency_ms: int
    cost_usd: float

stages: List[PipelineStage] = [
    PipelineStage("ingest", "raw file (PDF/image)", "normalized image",
                  "Convert to standard format, deskew, enhance", 200, 0.001),
    PipelineStage("classify", "normalized image", "DocType + confidence",
                  "Determine document type for downstream schema", 300, 0.003),
    PipelineStage("layout_analysis", "normalized image", "List[Region]",
                  "Detect headers, tables, key-value pairs, signatures", 500, 0.005),
    PipelineStage("vlm_extract", "image + regions + schema", "Dict[str, Any]",
                  "Extract structured fields using VLM with few-shot prompt", 2000, 0.02),
    PipelineStage("validate", "extracted fields + rules", "ValidationReport",
                  "Check arithmetic, formats, required fields, cross-references", 50, 0.001),
    PipelineStage("route", "ValidationReport + confidence", "RouteDecision",
                  "Determine human review level based on confidence", 10, 0.0),
]

print("Document processing pipeline — stage overview")
print("=" * 75)
print(f"{'Stage':<18s} {'Input':>25s} → {'Output':<25s}  {'ms':>5s}  {'Cost':>6s}")
print("-" * 75)
total_ms = 0
total_cost = 0.0
for s in stages:
    print(f"{s.name:<18s} {s.input_type:>25s} → {s.output_type:<25s}  {s.latency_ms:>5d}  ${s.cost_usd:.3f}")
    total_ms += s.latency_ms
    total_cost += s.cost_usd
print("-" * 75)
print(f"{'TOTAL':<18s} {'':>25s}   {'':25s}  {total_ms:>5d}  ${total_cost:.3f}")
print(f"\n✓ End-to-end: ~{total_ms/1000:.1f}s, ~${total_cost:.3f} per document")

## Section 2 — Document classification

Before extraction, we must determine the **document type** to select the correct
extraction schema. Each type has different expected fields:

| Type | Key fields |
|------|-----------|
| Invoice | invoice_number, vendor, line_items, subtotal, tax, total |
| Receipt | store, date, items, total, payment_method |
| Purchase Order | po_number, requester, items, delivery_date, approval |
| Form | form_type, fields (dynamic), signatures |
| Report | title, author, sections, date |

In [ ]:
# Document type classifier with expected fields per type

@dataclass
class DocumentSchema:
    """Schema defining expected fields for a document type."""
    doc_type: DocType
    required_fields: List[str]
    optional_fields: List[str]
    has_line_items: bool
    has_signatures: bool

schemas: Dict[DocType, DocumentSchema] = {
    DocType.INVOICE: DocumentSchema(
        doc_type=DocType.INVOICE,
        required_fields=["invoice_number", "vendor_name", "date", "subtotal", "tax", "total"],
        optional_fields=["po_reference", "payment_terms", "due_date", "currency"],
        has_line_items=True,
        has_signatures=False,
    ),
    DocType.RECEIPT: DocumentSchema(
        doc_type=DocType.RECEIPT,
        required_fields=["store_name", "date", "total"],
        optional_fields=["payment_method", "cashier", "store_address", "tax"],
        has_line_items=True,
        has_signatures=False,
    ),
    DocType.PURCHASE_ORDER: DocumentSchema(
        doc_type=DocType.PURCHASE_ORDER,
        required_fields=["po_number", "requester", "vendor", "date", "total"],
        optional_fields=["delivery_date", "shipping_address", "approval_status"],
        has_line_items=True,
        has_signatures=True,
    ),
    DocType.FORM: DocumentSchema(
        doc_type=DocType.FORM,
        required_fields=["form_type", "date"],
        optional_fields=["applicant_name", "reference_number"],
        has_line_items=False,
        has_signatures=True,
    ),
    DocType.REPORT: DocumentSchema(
        doc_type=DocType.REPORT,
        required_fields=["title", "author", "date"],
        optional_fields=["department", "version", "page_count"],
        has_line_items=False,
        has_signatures=False,
    ),
}

# ── Simple keyword-based classifier ──
CLASSIFICATION_KEYWORDS: Dict[DocType, List[str]] = {
    DocType.INVOICE: ["invoice", "bill to", "amount due", "payment terms", "inv-", "invoice number"],
    DocType.RECEIPT: ["receipt", "cashier", "change due", "thank you", "subtotal", "payment method"],
    DocType.PURCHASE_ORDER: ["purchase order", "po number", "requisition", "delivery date", "ship to"],
    DocType.FORM: ["application", "form", "please fill", "signature", "applicant"],
    DocType.REPORT: ["report", "executive summary", "findings", "recommendations", "prepared by"],
}

def classify_document(text: str) -> Tuple[DocType, float]:
    """Classify document type from text content using keyword matching."""
    text_lower = text.lower()
    scores: Dict[DocType, int] = {}
    for doc_type, keywords in CLASSIFICATION_KEYWORDS.items():
        scores[doc_type] = sum(1 for kw in keywords if kw in text_lower)
    best_type = max(scores, key=scores.get)
    total_kw = sum(scores.values())
    confidence = scores[best_type] / max(total_kw, 1)
    return best_type, round(confidence, 2)

# ── Test classification ──
test_docs = [
    ("Invoice #INV-2024-001\nBill To: Acme Corp\nAmount Due: $648.00\nPayment Terms: Net 30", "invoice"),
    ("RECEIPT\nStore: QuickMart #42\nCashier: Sarah\nTotal: $23.99\nPayment Method: Credit", "receipt"),
    ("Purchase Order PO-8801\nRequisition: Engineering Dept\nDelivery Date: 2024-04-01\nShip To: Warehouse B", "purchase_order"),
    ("APPLICATION FORM\nApplicant: John Doe\nPlease fill out all fields\nSignature: ___", "form"),
    ("Q3 Financial Report\nPrepared By: Finance Dept\nExecutive Summary\nFindings and Recommendations", "report"),
]

print("Document classification results")
print("=" * 65)
for text, expected in test_docs:
    predicted_type, conf = classify_document(text)
    match = "✓" if predicted_type.value == expected else "✗"
    print(f"  {match} Expected: {expected:<16s} Predicted: {predicted_type.value:<16s} Conf: {conf:.2f}")
    schema = schemas[predicted_type]
    print(f"    Required fields: {schema.required_fields}")

print(f"\n✓ {len(schemas)} document schemas defined")

## Section 3 — Layout analysis

Layout analysis detects **structural regions** within a document: headers, tables,
key-value pairs, and free text. This guides the VLM extraction by providing
spatial context.

A simple heuristic approach uses text patterns to identify region types:
- Lines with `:` → key-value pairs
- Lines with consistent column alignment → table rows
- Large/bold text → headers
- Blocks of flowing text → paragraphs

In [ ]:
# Heuristic layout analysis — detect regions from text structure

class RegionType(Enum):
    """Types of document regions."""
    HEADER = "header"
    KEY_VALUE = "key_value"
    TABLE = "table"
    PARAGRAPH = "paragraph"
    SIGNATURE = "signature"

@dataclass
class DocumentRegion:
    """A detected region within a document."""
    region_type: RegionType
    content: str
    line_start: int
    line_end: int
    confidence: float

def analyze_layout(text: str) -> List[DocumentRegion]:
    """Detect document regions using heuristic rules."""
    lines = text.strip().split("\n")
    regions: List[DocumentRegion] = []
    i = 0

    while i < len(lines):
        line = lines[i].strip()
        if not line:
            i += 1
            continue

        # Header: short line, all caps or ends with common header words
        if len(line) < 50 and (line.isupper() or line.startswith("#")):
            regions.append(DocumentRegion(RegionType.HEADER, line, i, i, 0.85))
            i += 1
        # Key-value: contains colon separator
        elif ":" in line and len(line.split(":")[0]) < 30:
            kv_lines = [line]
            j = i + 1
            while j < len(lines) and ":" in lines[j] and len(lines[j].split(":")[0]) < 30:
                kv_lines.append(lines[j].strip())
                j += 1
            regions.append(DocumentRegion(
                RegionType.KEY_VALUE, "\n".join(kv_lines), i, j - 1, 0.80
            ))
            i = j
        # Table: contains pipe separators or tab-aligned columns
        elif "|" in line or "\t" in line:
            table_lines = [line]
            j = i + 1
            while j < len(lines) and ("|" in lines[j] or "---" in lines[j]):
                table_lines.append(lines[j].strip())
                j += 1
            regions.append(DocumentRegion(
                RegionType.TABLE, "\n".join(table_lines), i, j - 1, 0.75
            ))
            i = j
        # Signature line
        elif "___" in line or "signature" in line.lower():
            regions.append(DocumentRegion(RegionType.SIGNATURE, line, i, i, 0.70))
            i += 1
        # Paragraph: everything else
        else:
            para_lines = [line]
            j = i + 1
            while j < len(lines) and lines[j].strip() and ":" not in lines[j] and "|" not in lines[j]:
                para_lines.append(lines[j].strip())
                j += 1
            regions.append(DocumentRegion(
                RegionType.PARAGRAPH, "\n".join(para_lines), i, j - 1, 0.60
            ))
            i = j

    return regions

# ── Test layout analysis on a sample invoice ──
sample_text = """INVOICE

Invoice Number: INV-2024-001
Date: 2024-03-15
Vendor: Acme Supplies Inc.
Bill To: TechCorp LLC

| Item          | Qty | Unit Price | Total    |
|---------------|-----|-----------|----------|
| Widget A      | 10  | $25.00    | $250.00  |
| Widget B      | 5   | $50.00    | $250.00  |
| Service Fee   | 1   | $100.00   | $100.00  |

Subtotal: $600.00
Tax (8%): $48.00
Total: $648.00

Payment Terms: Net 30
Due Date: 2024-04-14

Authorized Signature: _______________
"""

regions = analyze_layout(sample_text)

print("Layout analysis results")
print("=" * 65)
for r in regions:
    preview = r.content[:60].replace("\n", " ↵ ")
    print(f"  [{r.region_type.value:<12s}] lines {r.line_start:>2d}-{r.line_end:>2d}  conf={r.confidence:.2f}  '{preview}…'")

print(f"\n✓ Detected {len(regions)} regions in sample invoice")
region_counts = {}
for r in regions:
    region_counts[r.region_type.value] = region_counts.get(r.region_type.value, 0) + 1
for rt, count in region_counts.items():
    print(f"  {rt}: {count}")

## Section 4 — VLM extraction design

The core extraction uses a **Vision Language Model** (e.g., GPT-4o) with a carefully
designed prompt. The prompt contains:

1. **System role** — "You are a document extraction specialist"
2. **Output schema** — JSON structure with all expected fields
3. **Few-shot examples** — 1-2 examples of correct extractions
4. **Confidence instructions** — "Rate your confidence per field 0-1"

The image is sent alongside the text prompt, enabling the VLM to use both visual
layout cues and text content.

In [ ]:
# VLM extraction prompt design

@dataclass
class ExtractionPrompt:
    """Prompt template for VLM-based field extraction."""
    system_message: str
    output_schema: Dict[str, Any]
    few_shot_examples: List[Dict[str, Any]]
    confidence_instruction: str

    def build_messages(self, doc_type: str) -> List[Dict[str, str]]:
        """Build the message list for the OpenAI API."""
        schema_str = json.dumps(self.output_schema, indent=2)
        examples_str = "\n\n".join(
            f"Example {i+1}:\n{json.dumps(ex, indent=2)}"
            for i, ex in enumerate(self.few_shot_examples)
        )
        user_content = (
            f"Extract all fields from this {doc_type} document.\n\n"
            f"Output JSON schema:\n```json\n{schema_str}\n```\n\n"
            f"Few-shot examples:\n{examples_str}\n\n"
            f"{self.confidence_instruction}"
        )
        return [
            {"role": "system", "content": self.system_message},
            {"role": "user", "content": user_content},
        ]

# ── Invoice extraction prompt ──
invoice_prompt = ExtractionPrompt(
    system_message=(
        "You are an expert document extraction specialist. "
        "Extract structured data from document images with high accuracy. "
        "Return ONLY valid JSON matching the provided schema. "
        "Include a confidence score (0.0-1.0) for each field."
    ),
    output_schema={
        "invoice_number": {"type": "string", "description": "Unique invoice identifier"},
        "vendor_name": {"type": "string", "description": "Name of the vendor/supplier"},
        "date": {"type": "string", "format": "YYYY-MM-DD"},
        "line_items": [
            {"description": "str", "qty": "int", "unit_price": "float", "total": "float"}
        ],
        "subtotal": {"type": "number"},
        "tax_rate": {"type": "number", "description": "Tax rate as decimal (e.g., 0.08)"},
        "tax": {"type": "number"},
        "total": {"type": "number"},
        "confidence": {"type": "object", "description": "Per-field confidence scores 0-1"},
    },
    few_shot_examples=[
        {
            "invoice_number": "INV-2023-099",
            "vendor_name": "SupplyChain Co.",
            "date": "2023-12-01",
            "line_items": [
                {"description": "Paper A4", "qty": 20, "unit_price": 5.00, "total": 100.00}
            ],
            "subtotal": 100.00,
            "tax_rate": 0.10,
            "tax": 10.00,
            "total": 110.00,
            "confidence": {
                "invoice_number": 0.98, "vendor_name": 0.95, "date": 0.99,
                "line_items": 0.90, "subtotal": 0.97, "tax": 0.95, "total": 0.98
            }
        }
    ],
    confidence_instruction=(
        "For each field, provide a confidence score between 0.0 and 1.0. "
        "Use 0.95+ for clearly visible fields, 0.7-0.95 for partially visible or "
        "inferred fields, and below 0.7 for uncertain extractions."
    ),
)

messages = invoice_prompt.build_messages("invoice")

print("VLM extraction prompt design")
print("=" * 60)
print(f"\nSystem message ({len(messages[0]['content'])} chars):")
print(f"  {messages[0]['content'][:100]}…")
print(f"\nUser prompt ({len(messages[1]['content'])} chars):")
for line in messages[1]["content"].split("\n")[:8]:
    print(f"  {line}")
print("  …")
print(f"\n✓ Prompt includes schema, {len(invoice_prompt.few_shot_examples)} few-shot example(s), and confidence instructions")
print(f"  Total prompt tokens (est): ~{sum(len(m['content']) for m in messages) // 4}")

## Section 5 — Validation pipeline

After VLM extraction, a **rule engine** validates the output. Rules are organized
by category:

1. **Type checks** — Is `invoice_number` a string? Is `total` a number?
2. **Format checks** — Does `date` match YYYY-MM-DD? Does `total` have ≤ 2 decimals?
3. **Arithmetic checks** — Do line items sum to subtotal? Does tax = subtotal × rate?
4. **Cross-reference checks** — Does PO number match an existing PO?
5. **Required field checks** — Are all required fields present and non-empty?

In [ ]:
# Rule-based validation engine

class Severity(Enum):
    ERROR = "error"
    WARNING = "warning"
    INFO = "info"

@dataclass
class ValidationRule:
    """A single validation rule."""
    rule_id: str
    category: str
    description: str
    severity: Severity
    check_fn: Optional[Callable] = None  # actual function set at runtime

@dataclass
class ValidationIssue:
    """An issue found during validation."""
    rule_id: str
    severity: Severity
    field: str
    message: str

def check_required_fields(data: Dict[str, Any], required: List[str]) -> List[ValidationIssue]:
    """Check that all required fields are present and non-empty."""
    issues: List[ValidationIssue] = []
    for field_name in required:
        val = data.get(field_name)
        if val is None or (isinstance(val, str) and val.strip() == ""):
            issues.append(ValidationIssue(
                "REQ-001", Severity.ERROR, field_name,
                f"Required field '{field_name}' is missing or empty"
            ))
    return issues

def check_arithmetic(data: Dict[str, Any]) -> List[ValidationIssue]:
    """Check arithmetic consistency of invoice fields."""
    issues: List[ValidationIssue] = []
    line_items = data.get("line_items", [])

    # Line item totals
    for i, item in enumerate(line_items):
        expected = item.get("qty", 0) * item.get("unit_price", 0)
        actual = item.get("total", 0)
        if abs(expected - actual) > 0.01:
            issues.append(ValidationIssue(
                "ARITH-001", Severity.ERROR, f"line_items[{i}].total",
                f"Expected {expected:.2f}, got {actual:.2f}"
            ))

    # Subtotal
    expected_sub = sum(item.get("total", 0) for item in line_items)
    actual_sub = data.get("subtotal", 0)
    if abs(expected_sub - actual_sub) > 0.01:
        issues.append(ValidationIssue(
            "ARITH-002", Severity.ERROR, "subtotal",
            f"Expected {expected_sub:.2f} (sum of line totals), got {actual_sub:.2f}"
        ))

    # Tax
    tax_rate = data.get("tax_rate", 0)
    expected_tax = round(actual_sub * tax_rate, 2)
    actual_tax = data.get("tax", 0)
    if abs(expected_tax - actual_tax) > 0.01:
        issues.append(ValidationIssue(
            "ARITH-003", Severity.WARNING, "tax",
            f"Expected {expected_tax:.2f} ({actual_sub:.2f} × {tax_rate}), got {actual_tax:.2f}"
        ))

    # Grand total
    expected_total = actual_sub + actual_tax
    actual_total = data.get("total", 0)
    if abs(expected_total - actual_total) > 0.01:
        issues.append(ValidationIssue(
            "ARITH-004", Severity.ERROR, "total",
            f"Expected {expected_total:.2f} ({actual_sub:.2f} + {actual_tax:.2f}), got {actual_total:.2f}"
        ))

    return issues

def check_formats(data: Dict[str, Any]) -> List[ValidationIssue]:
    """Check field format validity."""
    issues: List[ValidationIssue] = []
    date_val = data.get("date", "")
    if date_val and not re.match(r"^\d{4}-\d{2}-\d{2}$", date_val):
        issues.append(ValidationIssue(
            "FMT-001", Severity.WARNING, "date",
            f"Date '{date_val}' does not match YYYY-MM-DD format"
        ))
    total_val = data.get("total", 0)
    if isinstance(total_val, (int, float)) and total_val < 0:
        issues.append(ValidationIssue(
            "FMT-002", Severity.ERROR, "total",
            "Total amount is negative"
        ))
    return issues

# ── Test the validator ──
good_data: Dict[str, Any] = {
    "invoice_number": "INV-2024-001",
    "vendor_name": "Acme Supplies",
    "date": "2024-03-15",
    "line_items": [
        {"description": "Widget A", "qty": 10, "unit_price": 25.0, "total": 250.0},
        {"description": "Widget B", "qty": 5, "unit_price": 50.0, "total": 250.0},
    ],
    "subtotal": 500.0,
    "tax_rate": 0.08,
    "tax": 40.0,
    "total": 540.0,
}

bad_data: Dict[str, Any] = {
    "invoice_number": "",
    "vendor_name": "Acme Supplies",
    "date": "15/03/2024",
    "line_items": [
        {"description": "Widget A", "qty": 10, "unit_price": 25.0, "total": 300.0},
    ],
    "subtotal": 300.0,
    "tax_rate": 0.08,
    "tax": 30.0,
    "total": 350.0,
}

required = ["invoice_number", "vendor_name", "date", "subtotal", "tax", "total"]

for label, data in [("Good extraction", good_data), ("Bad extraction", bad_data)]:
    issues: List[ValidationIssue] = []
    issues.extend(check_required_fields(data, required))
    issues.extend(check_arithmetic(data))
    issues.extend(check_formats(data))

    print(f"\n{label}")
    print("=" * 60)
    if not issues:
        print("  ✓ All checks passed")
    for issue in issues:
        icon = "✗" if issue.severity == Severity.ERROR else "⚠"
        print(f"  {icon} [{issue.rule_id}] {issue.field}: {issue.message}")
    print(f"  Total issues: {len(issues)} ({sum(1 for i in issues if i.severity == Severity.ERROR)} errors)")

## Section 6 — Confidence scoring and routing

The final pipeline stage computes an **aggregate confidence score** from two sources:

1. **Extraction confidence** — Per-field scores from the VLM (0–1)
2. **Validation confidence** — Fraction of rules passed (0–1)

The combined score drives the routing decision:

```
combined = w_extract × avg(field_confidences) + w_validate × validation_pass_rate
```

Where `w_extract = 0.6` and `w_validate = 0.4` (validation failures are strong
negative signals).

In [ ]:
# Confidence scoring and routing logic

@dataclass
class ConfidenceReport:
    """Aggregated confidence scores for a document extraction."""
    doc_id: str
    field_confidences: Dict[str, float]
    extraction_confidence: float
    validation_pass_rate: float
    combined_confidence: float
    route: str
    detail: str

def compute_confidence(
    doc_id: str,
    field_confidences: Dict[str, float],
    n_rules_passed: int,
    n_rules_total: int,
    w_extract: float = 0.6,
    w_validate: float = 0.4,
) -> ConfidenceReport:
    """Compute combined confidence and routing decision."""
    extraction_conf = sum(field_confidences.values()) / max(len(field_confidences), 1)
    validation_rate = n_rules_passed / max(n_rules_total, 1)

    combined = w_extract * extraction_conf + w_validate * validation_rate

    if combined >= 0.95:
        route = "auto_approve"
        detail = "High confidence — auto-approved"
    elif combined >= 0.70:
        route = "quick_review"
        detail = "Medium confidence — flagged for quick review"
    else:
        route = "manual_entry"
        detail = "Low confidence — routed to manual entry"

    return ConfidenceReport(
        doc_id=doc_id,
        field_confidences=field_confidences,
        extraction_confidence=round(extraction_conf, 3),
        validation_pass_rate=round(validation_rate, 3),
        combined_confidence=round(combined, 3),
        route=route,
        detail=detail,
    )

# ── Test confidence scoring ──
test_cases = [
    ("DOC-001", {"invoice_number": 0.99, "vendor": 0.97, "total": 0.98, "date": 0.99}, 8, 8),
    ("DOC-002", {"invoice_number": 0.95, "vendor": 0.80, "total": 0.85, "date": 0.90}, 6, 8),
    ("DOC-003", {"invoice_number": 0.60, "vendor": 0.50, "total": 0.45, "date": 0.70}, 3, 8),
]

print("Confidence scoring and routing")
print("=" * 70)
for doc_id, fc, passed, total in test_cases:
    report = compute_confidence(doc_id, fc, passed, total)
    print(f"\n  {report.doc_id}:")
    print(f"    Extraction confidence : {report.extraction_confidence:.3f}")
    print(f"    Validation pass rate  : {report.validation_pass_rate:.3f}")
    print(f"    Combined confidence   : {report.combined_confidence:.3f}")
    print(f"    Route                 : {report.route}")
    print(f"    Detail                : {report.detail}")

print("\n✓ Routing thresholds: auto ≥ 0.95, review ≥ 0.70, manual < 0.70")

## Exercises

1. **Add receipt schema** — Design the full extraction prompt (system message, schema,
   few-shot example) for the `RECEIPT` document type. Include at least 5 required
   fields and 3 optional fields. Test the prompt builder.

2. **Custom validation rules** — Add two new validation rules: (a) check that the
   invoice date is not in the future, and (b) check that the vendor name has at
   least 2 characters. Integrate them into the validation pipeline.

3. **Weighted confidence** — Modify `compute_confidence` to weight critical fields
   (e.g., `total`, `invoice_number`) higher than optional fields. How does this
   change the routing distribution?

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | The pipeline has six stages: ingest → classify → layout → extract → validate → route |
| 2 | Document classification selects the extraction schema; wrong type = wrong fields |
| 3 | Layout analysis provides spatial context that improves extraction accuracy |
| 4 | VLM prompts need schema, few-shot examples, and explicit confidence instructions |
| 5 | Rule-based validation catches arithmetic, format, and completeness errors |
| 6 | Combined confidence (extraction + validation) drives automated routing decisions |

## What's Next

In **02 — Build**, we implement the full pipeline end-to-end: creating synthetic
documents with Pillow, classifying them, extracting fields with a VLM, validating
results, and routing decisions.

---

## References

1. OpenAI, *GPT-4 Vision System Card*, 2024 — <https://openai.com/research/gpt-4v-system-card>
2. Kim, G. et al., *OCR-Free Document Understanding Transformer (Donut)*, ECCV 2022.
3. Unstructured.io, *Open-Source Document Processing* — <https://unstructured.io/>
4. Microsoft, *Azure AI Document Intelligence* — <https://learn.microsoft.com/en-us/azure/ai-services/document-intelligence/>